In [1]:
import os,sys
import numpy as np
import pandas as pd
root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(root)
from utils import utils,models

In [2]:
device = utils.get_device()
print(f"Using device: {device}")
batch_size = 64

Using device: mps


In [3]:
model_name = 'ViT-B/16'
model, preprocess = utils.load_model(model_name, device)
dataset_dir = '../dataset/newsimages_test_and_evaluation_26_v1.0'
images_dir = f'{dataset_dir}/news_images_evaluation'
dataset = pd.read_csv(f"{dataset_dir}/news_articles_evaluation.csv",encoding="latin-1")
dataset.set_index('article_id', inplace=True)
dataset['article_title'] = dataset['article_title'].apply(utils.normalize_text)
records = [
    {
        "image_path": utils.make_path(images_dir, row['image_id']),
        "title": row['article_title']
    }
    for _, row in dataset.iterrows()
]
image_paths = [r["image_path"] for r in records]
model_path = f"../artifacts/{model_name.replace('/','_').lower()}_evaluation_image_embeddings.npy"
if os.path.exists(model_path):
    print("Using cached embeddings")
    image_embeddings = np.load(model_path)
else:
    image_embeddings = utils.encode_images(image_paths, model, preprocess, batch_size, device)
    np.save(model_path, image_embeddings)

Using cached embeddings


In [4]:
generator = models.OllamaPipeline()

In [5]:
id_to_index = utils.build_id_to_index(image_paths)
article_titles,ground_truth = utils.build_ground_truth(dataset,images_dir,id_to_index)
data = [(q, img_id[0]) for q, img_id in ground_truth.items()]

In [6]:
generation_clip  = models.QERetrieval(model,image_embeddings,generator,device,"../dataset/title_to_expanded_mapping.json")
mrr = utils.compute_mrr(generation_clip,data,10)
queries = list(ground_truth.keys())
recalls = utils.evaluate(generation_clip,queries,ground_truth)

expanded version cached
expanded version cached
expanded version cached
expanded version cached
expanded version cached
expanded version cached
expanded version cached
expanded version cached
expanded version cached
expanded version cached
expanded version cached
expanded version cached
expanded version cached
expanded version cached
expanded version cached
expanded version cached
expanded version cached
expanded version cached
expanded version cached
expanded version cached
expanded version cached
expanded version cached
expanded version cached
expanded version cached
expanded version cached
expanded version cached
expanded version cached
expanded version cached
expanded version cached
expanded version cached
expanded version cached
expanded version cached
expanded version cached
expanded version cached
expanded version cached
expanded version cached
expanded version cached
expanded version cached
expanded version cached
expanded version cached
expanded version cached
expanded version

In [7]:
mrr

np.float64(0.2746351286029048)

In [8]:
recalls

{1: np.float64(0.1891643059490085),
 5: np.float64(0.3880311614730878),
 10: np.float64(0.4847025495750708)}